# ImageNet-20 Classification — Colab training

Train a from-scratch ResNet on the 20-class ImageNet subset, evaluate, then export to ONNX. Run all cells in order.

**Before running:** upload the repo files and the unzipped datasets to Colab, then fill in the paths below.

**NPU steps (NOT in Colab):** compile ONNX -> `.mxq` with Qubee (Docker, `compile_mxq.py`) and run inference with MACCEL (`npu_infer.py`) on the lab Windows machine.

## 0. Setup & configure paths

In [ ]:
import os
import sys
import torch

# --- Fill in the locations on your Colab drive / workspace ---
PROJECT_DIR = "/content/classification"            # where dataset.py, model.py etc. live
TRAIN_TXT   = "dataset/imagenet_train20.txt"
TRAIN_DIR   = "/content/dataset/imagenet_train20a"
VAL_TXT     = "dataset/imagenet_val20.txt"
VAL_DIR     = "/content/dataset/imagenet_val20"
# ------------------------------------------------------------

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 0b. Load the classification folder from Google Drive

Run this once to mount Drive and copy the `classification` folder into the Colab VM. Edit `DRIVE_PROJECT_DIR` to the folder's path inside Drive (e.g. `MyDrive/data/classification` or just `MyDrive/classification`).

After copying, edit the four `*_DIR`/`*_TXT` paths in cell 0 if the folder layout differs.

In [ ]:
from google.colab import drive
import os, shutil, zipfile

drive.mount('/content/drive')

# --- Set the path of your classification folder INSIDE Drive ---
DRIVE_PROJECT_DIR = 'MyDrive/classification'
# -----------------------------------------------------

src = '/content/drive/' + DRIVE_PROJECT_DIR
dst = '/content/classification'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('Copied classification -> /content/classification')

# Unzip the datasets if the zips are present (and not yet extracted)
dataset_dir = os.path.join(dst, 'dataset')
for z in ['imagenet_train20.zip', 'imagenet_val20.zip']:
    zp = os.path.join(dataset_dir, z)
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(dataset_dir)
        print(f'Unzipped {z}')

print('Contents of /content/classification/dataset:', os.listdir(dataset_dir))

## 1. Train the model

Prints train & validation accuracy for every epoch and saves the best checkpoint to `checkpoint.pth`.

In [ ]:
!python train.py --train-txt {TRAIN_TXT} --train-dir {TRAIN_DIR} --val-txt {VAL_TXT} --val-dir {VAL_DIR} --epochs 40 --batch-size 64 --lr 0.05 --out checkpoint.pth

## 2. Per-class & overall accuracy (Step 3)

In [ ]:
!python eval.py --checkpoint checkpoint.pth --txt {VAL_TXT} --dir {VAL_DIR}

## 3. Export ONNX

Creates `model.onnx` (input `[1,3,240,240]`, output 20 classes). Normalization/transpose to HWC is baked in at Qubee compile time, so the ONNX holds the raw torch model (normalized NCHW input) exactly as trained.

In [ ]:
!python to_onnx.py --checkpoint checkpoint.pth --out model.onnx

## 4. Prepare calibration images

Pulls a balanced subset of the **training** split into a flat `calib_images/` folder (calibration data must be part of the training dataset).

In [ ]:
!python collect_calib_images.py --train-txt {TRAIN_TXT} --train-dir {TRAIN_DIR} --images-dir calib_images --num-calib 128print("Calibration images ready -> zip calib_images/ for the lab machine.")

## 5. Download artifacts

Download these to your machine (or upload to the NPU lab machine's `C:\NPU_Projects`):
- `model.onnx`
- `calib_images/` (or the Qubee `calib_data/` folder — see step 7)
- `checkpoint.pth` (submitted for the architecture/weights requirement)

In [ ]:
from google.colab import files
import zipfile
if not os.path.exists("calib_images.zip"):
    with zipfile.ZipFile("calib_images.zip", "w") as z:
        for fn in sorted(os.listdir("calib_images")):
            z.write(os.path.join("calib_images", fn))
for f in ["model.onnx", "calib_images.zip", "checkpoint.pth"]:
    if os.path.exists(f):
        files.download(f)
print("Done.")

## 6. (On NPU lab machine) Run the Qubee / MACCEL pipeline

After copying files to `C:\NPU_Projects` on the lab Windows machine (and having the Qubee wheel + MACCEL wheel there):

```bat
REM --- Step 4: start Qubee Docker container ---
docker run -it --name qb_compiler_container -v C:/NPU_Projects:/NPU_Projects mobilint/qbcompiler:0.12-cpu-ubuntu22.04 /bin/bash

REM --- Step 5: install Qubee inside container ---
pip install /NPU_Projects/qubee-0.12.0.0+aries2-py3-none-any.whl

REM --- Step 6: compile ONNX -> MXQ inside Docker ---
cd /NPU_Projects
python prepare_calib.py --train-txt imagenet_train20.txt --train-dir imagenet_train20a --images-dir calib_images --num-calib 128
python compile_mxq.py --onnx model.onnx --calib calib_data --out model.mxq
exit

REM --- Step 9: install MACCEL runtime on Windows ---
python -m pip install "C:\NPU_Projects\maccel-0.30.1-cp312-cp312-win_amd64.whl"

REM --- Step 10: run NPU inference & accuracy ---
python npu_infer.py --mxq model.mxq --txt imagenet_val20.txt --dir imagenet_val20
```

Then submit `model.mxq`, `train.py`, `prepare_calib.py`/`compile_mxq.py`, `model.py`, `checkpoint.pth`, and REPORT.md on Canvas.